# Marketing Campaign Dataset - Combined Analysis

This notebook combines all marketing dataset analyses from Assignments 1-3.
- **Part 1**: Supervised Learning (Neural Network, SVM, KNN) — Assignment 1
- **Part 2**: Randomized Optimization (RHC, SA, GA for NN training) — Assignment 2
- **Part 3**: Unsupervised Learning (K-Means, EM, PCA, ICA, RP) + DR-enhanced NN — Assignment 3


---
# Part 1: Supervised Learning (Assignment 1)

Neural Network, SVM, and KNN on the marketing campaign dataset.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset
import time
from datetime import datetime
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score


In [ ]:


marketing_file = "./dataset/marketing_campaign.csv"

marketing_df = pd.read_csv("./dataset/marketing_campaign.csv", sep='\t', encoding="ascii")


In [ ]:
marketing_df.info()

In [ ]:
marketing_df.describe()

In [ ]:
# Fill null values with median
marketing_df['Income'] = marketing_df['Income'].fillna(marketing_df['Income'].median())

In [ ]:
# Dt_Customer ->datetime
marketing_df['Dt_Customer'] = pd.to_datetime(marketing_df['Dt_Customer'], format='%d-%m-%Y')

reference_date = datetime(2014, 7, 1)
marketing_df['Customer_Since_Months'] = (reference_date - marketing_df['Dt_Customer']).dt.days // 30

marketing_df.drop(columns=['Dt_Customer'], inplace=True)

In [ ]:
print(marketing_df['Education'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Education'], drop_first=False)

In [ ]:
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].replace({'Absurd': 'Other', 'YOLO': 'Other'})

print(marketing_df['Marital_Status'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Marital_Status'], drop_first=False)

In [ ]:
marketing_df['Age'] = 2015 - marketing_df['Year_Birth']

marketing_df.drop(columns=['Year_Birth'], inplace=True)


In [ ]:
irrelevant_columns = ['ID', 'NumDealsPurchases', 'Response']
marketing_df = marketing_df.drop(columns=irrelevant_columns)

In [ ]:
marketing_df['AcceptedAny'] = (
    (marketing_df['AcceptedCmp1'] == 1) | 
    (marketing_df['AcceptedCmp2'] == 1) | 
    (marketing_df['AcceptedCmp3'] == 1) | 
    (marketing_df['AcceptedCmp4'] == 1) | 
    (marketing_df['AcceptedCmp5'] == 1)
).astype(int)

marketing_df.drop(columns=['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5'], inplace=True)

print(marketing_df[['AcceptedAny']].value_counts())

In [ ]:
marketing_df['Education'] = marketing_df[['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD']].idxmax(axis=1)

marketing_df['Marital_Status'] = marketing_df[['Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 
                                               'Marital_Status_Divorced', 'Marital_Status_Widow', 'Marital_Status_Alone',
                                               'Marital_Status_Other']].idxmax(axis=1)
marketing_df.drop(columns=['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD',
                           'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 'Marital_Status_Divorced', 
                           'Marital_Status_Widow', 'Marital_Status_Alone', 'Marital_Status_Other'], inplace=True)

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
marketing_df['Education'] = marketing_df['Education'].str.replace('Education_', '')
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].str.replace('Marital_Status_', '')

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
marketing_df['Education'] = label_enc.fit_transform(marketing_df['Education'])
marketing_df['Marital_Status'] = label_enc.fit_transform(marketing_df['Marital_Status'])

print(marketing_df[['Education', 'Marital_Status']].value_counts())

In [ ]:
def split_data(marketing_df, method='train_val_test', k=5, val_size=0.2, test_size=0.2):
    
    X = marketing_df.drop(columns=['AcceptedAny']).values
    y = marketing_df['AcceptedAny'].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    if method == 'train_val_test':
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        val_size_adjusted = val_size / (1 - test_size) 
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=val_size_adjusted, random_state=42, stratify=y_train_val
        )

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

        print(f"Train: {X_train_tensor.shape[0]}, Val: {X_val_tensor.shape[0]}, Test: {X_test_tensor.shape[0]}")
        return X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor

    elif method == 'cross_val':
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        folds = []

        for train_index, test_index in skf.split(X, y):
            X_train_fold, X_test_fold = X[train_index], X[test_index]
            y_train_fold, y_test_fold = y[train_index], y[test_index]

            X_train_tensor = torch.tensor(X_train_fold, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).unsqueeze(1)
            X_test_tensor = torch.tensor(X_test_fold, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test_fold, dtype=torch.float32).unsqueeze(1)

            folds.append((X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor))

        print(f"Train: {folds[0][0].shape[0]}, Test: {folds[0][2].shape[0]}")
        return folds


In [ ]:

# Drop constant columns that cause NaN when StandardScaler divides by std=0
# Z_CostContact is always 3, Z_Revenue is always 11 — they carry no information
constant_cols = [col for col in marketing_df.columns if marketing_df[col].nunique() <= 1]
print(f"Dropping constant columns: {constant_cols}")
marketing_df.drop(columns=constant_cols, inplace=True)

In [ ]:
# Simple Split
X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

print("NaN in X_train:", torch.isnan(X_train).any().item())
print("Inf in X_train:", torch.isinf(X_train).any().item())


In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, activation_function=nn.ReLU()):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(X_train.shape[1], 64),
            activation_function,
            nn.Linear(64, 32),
            activation_function,
            nn.Linear(32, 16),
            activation_function,
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
num_epochs = 300
batch_size = 64
learning_rate = 0.001

In [ ]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)




In [ ]:

val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train_model(model, train_loader, val_loader, test_loader, num_epochs=200, lr=0.001):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.BCELoss()
    train_losses, val_losses, test_losses = [], [], []
    best_model = None
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                test_loss += loss.item()
        test_loss /= len(test_loader)
        test_losses.append(test_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

        if best_model:
            torch.save(best_model, "best_model.pth")

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Test Loss: {test_loss:.4f}")

    plt.figure(figsize=(10, 5))
    plt.title('Learning Curve - Neural Network: Loss/Error Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.plot(range(1, num_epochs + 1), train_losses, label='Train Loss')
    plt.plot(range(1, num_epochs + 1), val_losses, label='Val Loss')
    plt.plot(range(1, num_epochs + 1), test_losses, label='Test Loss')
    plt.legend()
    # plt.savefig("loss_all_full_nn_300epoch_400data.png")
    plt.show()
    
    

    return train_losses, val_losses, test_losses


In [ ]:
model = SimpleNN(activation_function=nn.ReLU())
start_time = time.time()
train_losses, val_losses, test_losses = train_model(model, train_loader, val_loader, test_loader, num_epochs=num_epochs, lr=0.001)
end_time = time.time()
training_time = end_time - start_time
print(f"Training Time: {training_time:.2f}")

In [ ]:
# test_losses_list = [test_losses_01, test_losses_001, test_losses_0001]

In [ ]:
# training time
training_time_nn = [[6.24, 100], [17.64, 300],[30.20, 500]]


In [ ]:
# x = np.arange(1, 301)


# # Plotting
# plt.figure(figsize=(8, 5))
# plt.title("Effect of Learning Rate In Neural Network")
# plt.xlabel("Iterations/Epochs")
# plt.ylabel("Values")
# plt.plot(x, test_losses_list[0], label="Learning Rate-0.01", linestyle="-", marker="o", markersize=2)
# plt.plot(x, test_losses_list[1], label="Learning Rate-0.001", linestyle="-", marker="s", markersize=2)
# plt.plot(x, test_losses_list[2], label="Learning Rate-0.0001", linestyle="-", marker="^", markersize=2)
# plt.legend()
# plt.savefig("nn_lr.png")
# plt.grid(True)

# # Show the plot
# plt.show()

In [ ]:

epochs = [item[1] for item in training_time_nn]
training_times = [item[0] for item in training_time_nn]

plt.figure(figsize=(8, 5)) 
plt.title("Neural Network Training Time vs Epochs")
plt.xlabel("Epochs")
plt.ylabel("Training Time (seconds)")
plt.plot(epochs, training_times, marker='o', linestyle='-', label="Training Time vs Epochs")
plt.legend()
plt.grid(True)
plt.savefig("nn_training_time.png") 
plt.show()

In [ ]:

def evaluate_model(model, test_loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            predictions = (outputs > 0.5).float() 
            y_true.extend(y_batch.numpy())
            y_pred.extend(predictions.numpy())

    return y_true, y_pred

In [ ]:
y_true, y_pred = evaluate_model(model, test_loader)

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

In [ ]:
folds = split_data(marketing_df, method='cross_val', k=5)

In [ ]:

def cross_val_evaluate(model_class, folds, num_epochs=300, lr=0.001, batch_size=64):
    fold_accuracies, fold_f1_scores, fold_precisions, fold_recalls = [], [], [], []
    total_training_time = 0

    for fold_idx, (X_train, y_train, X_test, y_test) in enumerate(folds):
        print(f"=== Fold {fold_idx + 1} ===")
        
        train_dataset = TensorDataset(X_train, y_train)
        test_dataset = TensorDataset(X_test, y_test)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        model = model_class()
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
        criterion = nn.BCELoss()

        start_time = time.time()

        for epoch in range(num_epochs):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

        end_time = time.time()
        total_training_time += (end_time - start_time)

        y_true, y_pred = [], []
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                outputs = model(X_batch)
                predictions = (outputs > 0.5).float()
                y_true.extend(y_batch.numpy())
                y_pred.extend(predictions.numpy())

        accuracy = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)

        fold_accuracies.append(accuracy)
        fold_f1_scores.append(f1)
        fold_precisions.append(precision)
        fold_recalls.append(recall)


    print(f"Average Accuracy: {sum(fold_accuracies) / len(fold_accuracies):.2f}")
    print(f"Average F1-Score: {sum(fold_f1_scores) / len(fold_f1_scores):.2f}")
    print(f"Average Precision: {sum(fold_precisions) / len(fold_precisions):.2f}")
    print(f"Average Recall: {sum(fold_recalls) / len(fold_recalls):.2f}")
    print(f"Total Training Time: {total_training_time:.2f} seconds")

In [ ]:
cross_val_evaluate(SimpleNN, folds, num_epochs=300, lr=0.001, batch_size=64)

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.svm import SVC


In [ ]:
svm_model = SVC(kernel='rbf', C=1000, probability=True) 

start_time = time.time()
svm_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()

svm_training_time = end_time - start_time


In [ ]:
y_val_pred = svm_model.predict(X_val_scaled)
y_test_pred = svm_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"SVM Accuracy: {accuracy:.2f}")
print(f"SVM F1-Score: {f1:.2f}")
print(f"SVM Precision: {precision:.2f}")
print(f"SVM Recall: {recall:.2f}")
print(f"SVM Training Time: {svm_training_time:.2f} seconds")

In [ ]:
train_sizes = [400, 800, 1344]
svm_results = []

for size in train_sizes:
    print(f"\nTraining SVM with {size} samples")
    
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    svm_model = SVC(kernel='rbf', C=100, probability=True)  # Best hyperparameters
    svm_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = svm_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = svm_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)

    svm_results.append((size, train_error, test_error))

    print(f"Training Error: {train_error:.2f}")
    print(f"Testing Error: {test_error:.2f}")

svm_results = np.array(svm_results)

plt.figure(figsize=(10, 5))
plt.title("SVM: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(svm_results[:, 0], svm_results[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(svm_results[:, 0], svm_results[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("svm_train_size_vs_error.png")
plt.show()

In [ ]:
train_sizes = [400, 800, 1344]
svm_results = []


for size in train_sizes:
    print(f"\nTraining SVM with {size} samples")
    
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    svm_model = SVC(kernel='rbf', C=100, probability=True) # best hyperparameters
    svm_model.fit(X_train_subset, y_train_subset.ravel())

    y_test_pred = svm_model.predict(X_test_scaled)
    
    accuracy = accuracy_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)

    svm_results.append((size, accuracy, f1, precision, recall, training_time))

    print(f"SVM Accuracy: {accuracy:.2f}")
    print(f"SVM F1-Score: {f1:.2f}")
    print(f"SVM Precision: {precision:.2f}")
    print(f"SVM Recall: {recall:.2f}")
    print(f"SVM Training Time: {training_time:.2f} seconds")

svm_results = np.array(svm_results)

plt.figure(figsize=(10, 5))
plt.title("SVM: Train Size vs Model Performance")
plt.xlabel("Training Size")
plt.ylabel("Score")
plt.plot(svm_results[:, 0], svm_results[:, 2], marker='s', linestyle='-', label="F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("svm_train_size_vs_performance.png")
plt.show()

In [ ]:
# ============================================================
# Decision Tree
# ============================================================

from sklearn.tree import DecisionTreeClassifier

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning: max_depth ---
max_depths = [5, 7, 10, 15, 20, None]
dt_results = []

for depth in max_depths:
    dt_model = DecisionTreeClassifier(max_depth=depth, random_state=42)

    start_time = time.time()
    dt_model.fit(X_train_scaled, y_train.ravel())
    end_time = time.time()
    training_time = end_time - start_time

    y_val_pred = dt_model.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred)

    dt_results.append({
        'max_depth': depth,
        'val_f1': val_f1,
        'training_time': training_time
    })

print("{:<12} {:<10} {:<15}".format("Max Depth", "F1-Score", "Training Time (s)"))
for r in dt_results:
    print("{:<12} {:<10.4f} {:<15.4f}".format(str(r['max_depth']), r['val_f1'], r['training_time']))

best_dt = max(dt_results, key=lambda x: x['val_f1'])
print(f"\nBest Max Depth: {best_dt['max_depth']}, Val F1: {best_dt['val_f1']:.4f}")

In [ ]:

# --- Evaluate best DT on test set ---
best_depth = best_dt['max_depth']
dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)

start_time = time.time()
dt_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
dt_training_time = end_time - start_time

y_test_pred = dt_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nDecision Tree (max_depth={best_depth}) Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {dt_training_time:.4f} seconds")

In [ ]:

# --- Train size vs error ---
train_sizes = [400, 800, 1344]
dt_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
    dt_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = dt_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = dt_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    dt_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

dt_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in dt_size_results])

plt.figure(figsize=(10, 5))
plt.title("Decision Tree: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(dt_size_results_arr[:, 0], dt_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(dt_size_results_arr[:, 0], dt_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("dt_train_size_vs_error.png")
plt.show()

In [ ]:
# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("Decision Tree: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in dt_size_results], [r['test_f1'] for r in dt_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("dt_train_size_vs_f1.png")
plt.show()

In [ ]:
# ============================================================
# XGBoost (Gradient Boosted Decision Tree)
# ============================================================

!pip install xgboost

from xgboost import XGBClassifier

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning ---
xgb_configs = [
    {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 9, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.1},
]

xgb_results = []

for cfg in xgb_configs:
    xgb_model = XGBClassifier(
        n_estimators=cfg['n_estimators'],
        max_depth=cfg['max_depth'],
        learning_rate=cfg['learning_rate'],
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )

    start_time = time.time()
    xgb_model.fit(X_train_scaled, y_train.ravel())
    end_time = time.time()
    training_time = end_time - start_time

    y_val_pred = xgb_model.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred)

    xgb_results.append({
        **cfg,
        'val_f1': val_f1,
        'training_time': training_time
    })

print("{:<15} {:<12} {:<15} {:<10} {:<15}".format(
    "n_estimators", "max_depth", "learning_rate", "F1-Score", "Time (s)"))
for r in xgb_results:
    print("{:<15} {:<12} {:<15} {:<10.4f} {:<15.4f}".format(
        r['n_estimators'], r['max_depth'], r['learning_rate'], r['val_f1'], r['training_time']))

best_xgb = max(xgb_results, key=lambda x: x['val_f1'])
print(f"\nBest Config: n_estimators={best_xgb['n_estimators']}, max_depth={best_xgb['max_depth']}, "
      f"lr={best_xgb['learning_rate']}, Val F1={best_xgb['val_f1']:.4f}")


In [ ]:

# --- Evaluate best XGBoost on test set ---
xgb_model = XGBClassifier(
    n_estimators=best_xgb['n_estimators'],
    max_depth=best_xgb['max_depth'],
    learning_rate=best_xgb['learning_rate'],
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

start_time = time.time()
xgb_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
xgb_training_time = end_time - start_time

y_test_pred = xgb_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nXGBoost Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {xgb_training_time:.4f} seconds")

In [ ]:

# --- Train size vs error ---
train_sizes = [400, 800, 1344]
xgb_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    xgb_model = XGBClassifier(
        n_estimators=best_xgb['n_estimators'],
        max_depth=best_xgb['max_depth'],
        learning_rate=best_xgb['learning_rate'],
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = xgb_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = xgb_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    xgb_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

xgb_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in xgb_size_results])

plt.figure(figsize=(10, 5))
plt.title("XGBoost: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(xgb_size_results_arr[:, 0], xgb_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(xgb_size_results_arr[:, 0], xgb_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("xgb_train_size_vs_error.png")
plt.show()

In [ ]:

# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("XGBoost: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in xgb_size_results], [r['test_f1'] for r in xgb_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("xgb_train_size_vs_f1.png")
plt.show()

# KNN

In [ ]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import time

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)



In [ ]:

k_values = [3, 5, 7, 9]
distance_metrics = ['manhattan', 'euclidean']
results = []

for k in k_values:
    for metric in distance_metrics:
        knn_model = KNeighborsClassifier(n_neighbors=k, metric=metric)
        
        start_time = time.time()
        knn_model.fit(X_train_scaled, y_train)
        end_time = time.time()
        training_time = end_time - start_time

        y_val_pred = knn_model.predict(X_val_scaled)
        val_f1 = f1_score(y_val, y_val_pred)

        results.append({
            'k': k,
            'metric': metric,
            'val_f1': val_f1,
            'training_time': training_time
        })

print("{:<10} {:<15} {:<10} {:<15}".format("K", "Distance Metric", "F1-Score", "Training Time (s)"))
for result in results:
    print("{:<10} {:<15} {:<10.4f} {:<15.2f}".format(result['k'], result['metric'], result['val_f1'], result['training_time']))

best_result = max(results, key=lambda x: x['val_f1'])
print("\nBest Parameters:")
print(f"K: {best_result['k']}, Distance Metric: {best_result['metric']}, F1-Score: {best_result['val_f1']:.4f}, Training Time: {best_result['training_time']:.2f} seconds")

In [ ]:
train_sizes = [400, 800, 1344]
results = []

for size in train_sizes:
    X_train_partial = X_train_scaled[:size]
    y_train_partial = y_train[:size]

    knn_model = KNeighborsClassifier(n_neighbors=3, metric='euclidean')

    start_time = time.time()
    knn_model.fit(X_train_partial, y_train_partial)
    end_time = time.time()
    training_time = end_time - start_time

    y_train_pred = knn_model.predict(X_train_partial)
    y_test_pred = knn_model.predict(X_test_scaled)

    train_loss = 1 - accuracy_score(y_train_partial, y_train_pred)
    test_loss = 1 - accuracy_score(y_test, y_test_pred)

    results.append({'train_size': size, 'train_loss': train_loss, 'test_loss': test_loss})

train_sizes = [r['train_size'] for r in results]
train_losses = [r['train_loss'] for r in results]
test_losses = [r['test_loss'] for r in results]

# Plot
plt.figure(figsize=(10, 5))
plt.title("KNN: Train Size vs Training and Testing Loss")
plt.xlabel("Training Size")
plt.ylabel("Loss")
plt.plot(train_sizes, train_losses, marker='o', linestyle='-', label="Training Loss")
plt.plot(train_sizes, test_losses, marker='s', linestyle='-', label="Testing Loss")
plt.legend()
plt.grid(True)
plt.savefig("knn_train_size_vs_loss.png")
plt.show()

In [ ]:
train_sizes = [400, 800, 1344]
results = []

for size in train_sizes:
    X_train_partial = X_train_scaled[:size]
    y_train_partial = y_train[:size]

    knn_model = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
    
    start_time = time.time()
    knn_model.fit(X_train_partial, y_train_partial)
    end_time = time.time()
    training_time = end_time - start_time

    y_val_pred = knn_model.predict(X_val_scaled)
    y_test_pred = knn_model.predict(X_test_scaled)

    val_f1 = f1_score(y_val, y_val_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)

    results.append({'train_size': size,'test_f1': test_f1,})

train_sizes = [r['train_size'] for r in results]
test_f1_scores = [r['test_f1'] for r in results]

plt.title("KNN: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot(train_sizes, test_f1_scores, marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("knn_train_size_vs_f1.png")
plt.show()

---
# Part 2: Randomized Optimization (Assignment 2)

Using Random Hill Climbing (RHC), Simulated Annealing (SA), and Genetic Algorithm (GA) 
to train Neural Network weights on the marketing campaign dataset, compared with SGD.

*(K-Coloring and TSP optimization problems are excluded as they are not related to the marketing dataset.)*


In [ ]:
!pip install joblib==1.2.0

In [ ]:
!pip install mlrose-hiive

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset
import time
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score


In [ ]:
import mlrose_hiive as mlrose

In [ ]:
np.random.seed(42) # random seed

In [ ]:
marketing_file = "./dataset/marketing_campaign.csv"
marketing_df = pd.read_csv("./dataset/marketing_campaign.csv", sep='\t', encoding="ascii")

marketing_df.info()

In [ ]:
marketing_df.describe()

In [ ]:
# Fill null values with median
marketing_df['Income'].fillna(marketing_df['Income'].median())

In [ ]:
# Dt_Customer ->datetime
marketing_df['Dt_Customer'] = pd.to_datetime(marketing_df['Dt_Customer'], format='%d-%m-%Y')

reference_date = datetime(2014, 7, 1)
marketing_df['Customer_Since_Months'] = (reference_date - marketing_df['Dt_Customer']).dt.days // 30

marketing_df.drop(columns=['Dt_Customer'], inplace=True)

print(marketing_df['Education'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Education'], drop_first=False)

In [ ]:
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].replace({'Absurd': 'Other', 'YOLO': 'Other'})

print(marketing_df['Marital_Status'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Marital_Status'], drop_first=False)

In [ ]:
marketing_df['Age'] = 2015 - marketing_df['Year_Birth']

marketing_df.drop(columns=['Year_Birth'], inplace=True)

irrelevant_columns = ['ID', 'NumDealsPurchases', 'Response']
marketing_df = marketing_df.drop(columns=irrelevant_columns)

marketing_df['AcceptedAny'] = (
    (marketing_df['AcceptedCmp1'] == 1) |
    (marketing_df['AcceptedCmp2'] == 1) |
    (marketing_df['AcceptedCmp3'] == 1) |
    (marketing_df['AcceptedCmp4'] == 1) |
    (marketing_df['AcceptedCmp5'] == 1)
).astype(int)

marketing_df.drop(columns=['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5'], inplace=True)

print(marketing_df[['AcceptedAny']].value_counts())

marketing_df['Education'] = marketing_df[['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD']].idxmax(axis=1)

marketing_df['Marital_Status'] = marketing_df[['Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married',
                                               'Marital_Status_Divorced', 'Marital_Status_Widow', 'Marital_Status_Alone',
                                               'Marital_Status_Other']].idxmax(axis=1)
marketing_df.drop(columns=['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD',
                           'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 'Marital_Status_Divorced',
                           'Marital_Status_Widow', 'Marital_Status_Alone', 'Marital_Status_Other'], inplace=True)

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
marketing_df['Education'] = marketing_df['Education'].str.replace('Education_', '')
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].str.replace('Marital_Status_', '')

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
marketing_df['Education'] = label_enc.fit_transform(marketing_df['Education'])
marketing_df['Marital_Status'] = label_enc.fit_transform(marketing_df['Marital_Status'])

print(marketing_df[['Education', 'Marital_Status']].value_counts())

In [ ]:

# Drop constant columns that cause NaN when StandardScaler divides by std=0
# Z_CostContact is always 3, Z_Revenue is always 11 — they carry no information
constant_cols = [col for col in marketing_df.columns if marketing_df[col].nunique() <= 1]
print(f"Dropping constant columns: {constant_cols}")
marketing_df.drop(columns=constant_cols, inplace=True)

In [ ]:
def split_data(marketing_df, method='train_val_test', k=5, val_size=0.2, test_size=0.2):

    X = marketing_df.drop(columns=['AcceptedAny']).values
    y = marketing_df['AcceptedAny'].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    if method == 'train_val_test':
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )

        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=val_size_adjusted, random_state=42, stratify=y_train_val
        )

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

        print(f"Train: {X_train_tensor.shape[0]}, Val: {X_val_tensor.shape[0]}, Test: {X_test_tensor.shape[0]}")
        return X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor

    elif method == 'cross_val':
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        folds = []

        for train_index, test_index in skf.split(X, y):
            X_train_fold, X_test_fold = X[train_index], X[test_index]
            y_train_fold, y_test_fold = y[train_index], y[test_index]

            X_train_tensor = torch.tensor(X_train_fold, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).unsqueeze(1)
            X_test_tensor = torch.tensor(X_test_fold, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test_fold, dtype=torch.float32).unsqueeze(1)

            folds.append((X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor))

        print(f"Train: {folds[0][0].shape[0]}, Test: {folds[0][2].shape[0]}")
        return folds


In [ ]:
# Simple Split
X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')


In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, activation_function=nn.ReLU()):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(X_train.shape[1], 64),
            activation_function,
            nn.Linear(64, 32),
            activation_function,
            nn.Linear(32, 16),
            activation_function,
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
num_epochs = 300
batch_size = 64
learning_rate = 0.001

In [ ]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)



val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train_model(model, train_loader, val_loader, test_loader, num_epochs=200, lr=0.001):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.BCELoss()
    train_losses, val_losses, test_losses = [], [], []
    best_model = None
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                test_loss += loss.item()
        test_loss /= len(test_loader)
        test_losses.append(test_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

        if best_model:
            torch.save(best_model, "best_model.pth")

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Test Loss: {test_loss:.4f}")

    plt.figure(figsize=(10, 5))
    plt.title('Learning Curve - Neural Network: Loss/Error Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.plot(range(1, num_epochs + 1), train_losses, label='Train Loss')
    plt.plot(range(1, num_epochs + 1), val_losses, label='Val Loss')
    plt.plot(range(1, num_epochs + 1), test_losses, label='Test Loss')
    plt.legend()
    plt.savefig("loss_all_full_nn_300epoch.png")
    plt.show()



    return train_losses, -train_losses[-1]


In [ ]:
model_ro = SimpleNN(activation_function=nn.ReLU())


In [ ]:
def get_weights(model):
    return np.concatenate([param.detach().cpu().numpy().flatten() for param in model.parameters()])

def set_weights(model, flat_weights):
    pointer = 0
    for param in model.parameters():
        num_param = param.numel()
        new_weights = flat_weights[pointer:pointer+num_param].reshape(param.shape)
        param.data = torch.tensor(new_weights, dtype=param.data.dtype, device=param.data.device)
        pointer += num_param

def fitness_function(weights):
    set_weights(model_ro, weights)
    model_ro.eval()
    criterion = nn.BCELoss()
    with torch.no_grad():
        outputs = model_ro(X_train)
        loss = criterion(outputs, y_train).item()
    return -loss

In [ ]:
def train_with_RO(algorithm, max_iters=300, max_attempts=100):
    init_weights = get_weights(model_ro)
    fitness = mlrose.CustomFitness(fitness_function)
    problem = mlrose.ContinuousOpt(length=len(init_weights), fitness_fn=fitness, maximize=True, max_val=1, min_val=-1)

    if algorithm == "RHC":
        best_weights, best_fitness, fitness_curve = mlrose.random_hill_climb(
            problem, max_iters=max_iters, max_attempts=max_attempts, curve=True, random_state=42)
    elif algorithm == "SA":
        schedule = mlrose.GeomDecay(init_temp=1.0, decay=0.95, min_temp=0.001)
        best_weights, best_fitness, fitness_curve = mlrose.simulated_annealing(
            problem, schedule=schedule, max_iters=max_iters, max_attempts=max_attempts, curve=True, random_state=42)
    elif algorithm == "GA":
        best_weights, best_fitness, fitness_curve = mlrose.genetic_alg(
            problem, pop_size=200, mutation_prob=0.1, max_iters=max_iters, max_attempts=max_attempts, curve=True, random_state=42)
    else:
        raise ValueError("Invalid algorithm")
    return best_weights, best_fitness, fitness_curve

In [ ]:
model_sgd = SimpleNN(activation_function=nn.ReLU())
start_time = time.time()
sgd_losses, sgd_fitness = train_model(model_sgd, train_loader, val_loader, test_loader, num_epochs=num_epochs, lr=0.001)

end_time = time.time()
training_time = end_time - start_time



In [ ]:
ro_algorithms = ["RHC", "SA", "GA"]
ro_results = {}
for algo in ro_algorithms:
    model_ro = SimpleNN(activation_function=nn.ReLU())
    start = time.time()
    best_w, best_fit, fitness_curve = train_with_RO(algo, max_iters=300, max_attempts=100)
    elapsed = time.time() - start
    ro_results[algo] = {"fitness_curve": fitness_curve[:, 0], "time": elapsed, "final_fitness": best_fit}
    print(f"{algo} - Final Fitness: {best_fit:.4f}, Time: {elapsed:.4f}s")

plt.figure(figsize=(8,5))
iterations = np.arange(1, 301)
plt.plot(iterations, -np.array(sgd_losses), label='SGD', color='blue', linestyle='-')
for algo in ro_algorithms:
    plt.plot(range(1, len(ro_results[algo]["fitness_curve"])+1), ro_results[algo]["fitness_curve"], label=algo, linestyle='--')
plt.xlabel("Iterations/Epochs")
plt.ylabel("Fitness (-Loss)")
plt.title("Fitness vs Iterations (Neural Network Training)")
plt.legend()
plt.grid()
plt.savefig("nn_fitness_vs_iterations.png", dpi=300, bbox_inches="tight")
plt.show()

methods = ['SGD'] + ro_algorithms
times = [training_time] + [ro_results[algo]["time"] for algo in ro_algorithms]
plt.figure(figsize=(8,5))
plt.bar(methods, times, color=['blue', 'orange', 'green', 'red'])
plt.xlabel("Method")
plt.ylabel("Training Time (s)")
plt.title("Comparison of Training Time")
plt.savefig("nn_training_time_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

hidden_sizes = [16, 32, 64, 128]
sgd_fitness_vs_size = {}
ro_fitness_vs_size = {algo: {} for algo in ro_algorithms}

for size in hidden_sizes:
    model_temp = SimpleNN(activation_function=nn.ReLU())
    optimizer = optim.SGD(model_temp.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
    criterion = nn.BCELoss()
    losses = []
    for epoch in range(300):
        model_temp.train()
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model_temp(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(train_loader))
    sgd_fitness_vs_size[size] = -losses[-1]

    for algo in ro_algorithms:
        model_ro = SimpleNN(activation_function=nn.ReLU())
        init_weights = get_weights(model_ro)

        def fitness_nn(w):
            set_weights(model_ro, w)
            model_ro.eval()
            with torch.no_grad():
                outputs = model_ro(X_train)
                loss = criterion(outputs, y_train).item()
            return -loss
        fitness = mlrose.CustomFitness(fitness_nn)
        problem = mlrose.ContinuousOpt(length=len(init_weights), fitness_fn=fitness, maximize=True, max_val=1, min_val=-1)
        best_w, best_fit, _ = mlrose.genetic_alg(problem, max_iters=300, pop_size=200, mutation_prob=0.1, max_attempts=100, random_state=42)
        ro_fitness_vs_size[algo][size] = best_fit

plt.figure(figsize=(8,5))
plt.plot(hidden_sizes, [sgd_fitness_vs_size[size] for size in hidden_sizes], marker='o', label='SGD')
for algo in ro_algorithms:
    plt.plot(hidden_sizes, [ro_fitness_vs_size[algo][size] for size in hidden_sizes], marker='o', label=algo)
plt.xlabel("Hidden Layer Size")
plt.ylabel("Final Fitness (-Loss)")
plt.title("Fitness vs. Problem Size (Network Architecture)")
plt.legend()
plt.grid()
plt.savefig("nn_fitness_vs_size.png", dpi=300, bbox_inches="tight")
plt.show()

---
# Part 3: Unsupervised Learning (Assignment 3)

K-Means clustering, Expectation Maximization (EM/GMM), PCA, ICA, Random Projection,
and Neural Network training with dimensionality reduction and cluster features
on the marketing campaign dataset.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset
import time
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score


In [ ]:


marketing_file = "./dataset/marketing_campaign.csv"

marketing_df = pd.read_csv("./dataset/marketing_campaign.csv", sep='\t', encoding="ascii")


In [ ]:
marketing_df.info()

In [ ]:
marketing_df.describe()

In [ ]:
# Fill null values with median
marketing_df['Income'] = marketing_df['Income'].fillna(marketing_df['Income'].median())

In [ ]:
# Dt_Customer ->datetime
marketing_df['Dt_Customer'] = pd.to_datetime(marketing_df['Dt_Customer'], format='%d-%m-%Y')

reference_date = datetime(2014, 7, 1)
marketing_df['Customer_Since_Months'] = (reference_date - marketing_df['Dt_Customer']).dt.days // 30

marketing_df.drop(columns=['Dt_Customer'], inplace=True)

In [ ]:
print(marketing_df['Education'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Education'], drop_first=False)

In [ ]:
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].replace({'Absurd': 'Other', 'YOLO': 'Other'})

print(marketing_df['Marital_Status'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Marital_Status'], drop_first=False)

In [ ]:
marketing_df['Age'] = 2015 - marketing_df['Year_Birth']

marketing_df.drop(columns=['Year_Birth'], inplace=True)


In [ ]:
irrelevant_columns = ['ID']
marketing_df = marketing_df.drop(columns=irrelevant_columns)

In [ ]:

features = marketing_df.columns
scaler = StandardScaler()
scaled_features = scaler.fit_transform(marketing_df[features])


df_scaled = pd.DataFrame(scaled_features, columns=features)

In [ ]:
df_scaled

PRE DATA PROCESSING DONE.

In [ ]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = [col for col in df_scaled.columns]
X = df_scaled[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k_values = range(3, 11)
silhouette_scores = []
inertias = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    print(f"k = {k}: Inertia = {kmeans.inertia_:.2f}, Silhouette = {silhouette_score(X_scaled, labels):.4f}")

fig, ax1 = plt.subplots(figsize=(8,6))
color = 'tab:blue'
ax1.set_xlabel('k')
ax1.set_ylabel('Silhouette Score', color=color)
ax1.plot(k_values, silhouette_scores, marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(list(k_values))
ax1.set_title("K-Means Evaluation")

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Inertia', color=color)
ax2.plot(k_values, inertias, marker='x', linestyle='--', color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features = list(df_scaled.columns)
X = df_scaled[features]



k = 5
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(X)

df_scaled['Cluster'] = labels

cluster_stats = df_scaled.groupby('Cluster').mean()
print("Cluster Statistics:")
print(cluster_stats)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(8,6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', alpha=0.6)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means Clusters (k=5) in PCA Space')
plt.colorbar(label='Cluster')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
income_means = df_scaled.groupby('Cluster')['Income'].mean()
wine_means = df_scaled.groupby('Cluster')['MntWines'].mean()

income_means.plot(kind='bar', ax=ax[0])
ax[0].set_title('Average Income by Cluster')
ax[0].set_ylabel('Income (scaled)')

wine_means.plot(kind='bar', ax=ax[1])
ax[1].set_title('Average MntWines by Cluster')
ax[1].set_ylabel('MntWines (scaled)')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture

features = [col for col in marketing_df.columns]
X = marketing_df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 5
gmm = GaussianMixture(n_components=k, random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

df_scaled = pd.DataFrame(X_scaled, columns=features)
df_scaled['Cluster'] = gmm_labels

cluster_stats = df_scaled.groupby('Cluster').mean()
print("Cluster Statistics (means):")
print(cluster_stats)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=gmm_labels, cmap='viridis', alpha=0.6)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'EM (GMM) Clusters (k={k}) in PCA Space (Marketing Data)')
plt.colorbar(label='Cluster')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

income_means = df_scaled.groupby('Cluster')['Income'].mean()
wine_means = df_scaled.groupby('Cluster')['MntWines'].mean()

income_means.plot(kind='bar', ax=ax[0])
ax[0].set_title('Average Income by Cluster (GMM)')
ax[0].set_ylabel('Income (scaled)')

wine_means.plot(kind='bar', ax=ax[1])
ax[1].set_title('Average MntWines by Cluster (GMM)')
ax[1].set_ylabel('MntWines (scaled)')

plt.tight_layout()
plt.show()

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA
from sklearn.random_projection import GaussianRandomProjection
from scipy.stats import kurtosis

features = list(marketing_df.columns)
X = marketing_df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=15)
X_pca = pca.fit_transform(X_scaled)
eigenvalues = pca.explained_variance_
explained_ratio = pca.explained_variance_ratio_

plt.figure(figsize=(8,5))
plt.plot(np.cumsum(explained_ratio), marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('PCA: Cumulative Explained Variance')
plt.grid(True)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(range(1, 16), eigenvalues, color='steelblue', edgecolor='k')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue')
plt.title('PCA: Eigenvalue Distribution')
plt.show()

ica = FastICA(n_components=10, random_state=42)
X_ica = ica.fit_transform(X_scaled)
ica_kurtosis = kurtosis(X_ica, fisher=True)
print("Kurtosis of each ICA component:", ica_kurtosis)

plt.figure(figsize=(15,10))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.hist(X_ica[:, i], bins=30, color='skyblue', edgecolor='k')
    plt.title(f'ICA Component {i+1}\nKurtosis: {ica_kurtosis[i]:.2f}')
plt.tight_layout()
plt.show()

rp = GaussianRandomProjection(n_components=10, random_state=42)
X_rp = rp.fit_transform(X_scaled)
rp_reconstructed = np.dot(X_rp, np.linalg.pinv(rp.components_.T))
reconstruction_error = np.linalg.norm(X_scaled - rp_reconstructed, ord='fro')
print("Random Projection Reconstruction Error (Frobenius norm):", reconstruction_error)

errors = []
for seed in range(10):
    rp_temp = GaussianRandomProjection(n_components=10, random_state=seed)
    X_rp_temp = rp_temp.fit_transform(X_scaled)
    rp_rec_temp = np.dot(X_rp_temp, np.linalg.pinv(rp_temp.components_.T))
    err = np.linalg.norm(X_scaled - rp_rec_temp, ord='fro')
    errors.append(err)
print("Reconstruction errors from different random projections:", errors)

plt.figure(figsize=(8,5))
plt.bar(range(10), errors, color='coral', edgecolor='k')
plt.xlabel('Random Seed')
plt.ylabel('Reconstruction Error (Frobenius norm)')
plt.title('Random Projection Reconstruction Errors for Different Seeds')
plt.show()

# Data rank and collinearity analysis
data_rank = np.linalg.matrix_rank(X_scaled)
print("Rank of the data:", data_rank)
corr_matrix = np.corrcoef(X_scaled, rowvar=False)
avg_correlation = np.nanmean(np.abs(corr_matrix[np.triu_indices_from(corr_matrix, k=1)]))
print("Average absolute correlation between features:", avg_correlation)

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False)
plt.title('Correlation Matrix of Standardized Data')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ica_kurtosis = np.array([3.4379667, 2.80691173, -1.18911826, -1.68886734, 0.40161493,
                         -1.21420517, 15.49343555, -1.66802089, 0.27066432, 1.08586054])

plt.figure(figsize=(8,6))
plt.bar(range(1, len(ica_kurtosis)+1), ica_kurtosis, color='skyblue', edgecolor='k')
plt.xlabel("ICA Component")
plt.ylabel("Kurtosis")
plt.title("Kurtosis of Each ICA Component")
plt.xticks(range(1, len(ica_kurtosis)+1))
plt.axhline(0, color='gray', linewidth=0.8)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA
from sklearn.random_projection import GaussianRandomProjection
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

features = [col for col in marketing_df.columns]
X = marketing_df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


def evaluate_dr_clustering(X_data, dr_method, cluster_method, n_clusters=4):

    if dr_method == 'PCA':
        dr = PCA(n_components=10, random_state=42)
        X_reduced = dr.fit_transform(X_data)

    elif dr_method == 'ICA':
        dr = FastICA(n_components=10, random_state=42)
        X_reduced = dr.fit_transform(X_data)
    elif dr_method == 'RP':
        dr = GaussianRandomProjection(n_components=10, random_state=42)
        X_reduced = dr.fit_transform(X_data)
    else:
        raise ValueError("Unknown DR method.")

    if cluster_method == 'KMeans':
        clust = KMeans(n_clusters=n_clusters, random_state=42)
        labels = clust.fit_predict(X_reduced)
        inertia = clust.inertia_
        silhouette = silhouette_score(X_reduced, labels)
        return {
            'DR_Method': dr_method,
            'Clust_Algo': cluster_method,
            'Silhouette': silhouette,
            'Inertia': inertia,
            'LogLikelihood': None
        }
    elif cluster_method == 'EM':
        gm = GaussianMixture(n_components=n_clusters, random_state=42)
        gm.fit(X_reduced)
        labels = gm.predict(X_reduced)
        silhouette = silhouette_score(X_reduced, labels)
        log_likelihood = gm.lower_bound_
        return {
            'DR_Method': dr_method,
            'Clust_Algo': cluster_method,
            'Silhouette': silhouette,
            'Inertia': None,
            'LogLikelihood': log_likelihood
        }
    else:
        raise ValueError("Unknown clustering method.")


dr_methods = ['PCA', 'ICA', 'RP']
cluster_algos = ['KMeans', 'EM']
n_clusters = 5

results = []
for dr_m in dr_methods:
    for clust_m in cluster_algos:
        result_dict = evaluate_dr_clustering(X_scaled, dr_m, clust_m, n_clusters)
        results.append(result_dict)

df_results = pd.DataFrame(results)
print(df_results)



In [ ]:
def split_data(marketing_df, method='train_val_test', k=5, val_size=0.2, test_size=0.2):

    X = marketing_df.drop(columns=['AcceptedAny']).values
    y = marketing_df['AcceptedAny'].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    if method == 'train_val_test':
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )

        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=val_size_adjusted, random_state=42, stratify=y_train_val
        )

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

        print(f"Train: {X_train_tensor.shape[0]}, Val: {X_val_tensor.shape[0]}, Test: {X_test_tensor.shape[0]}")
        return X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor

    elif method == 'cross_val':
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        folds = []

        for train_index, test_index in skf.split(X, y):
            X_train_fold, X_test_fold = X[train_index], X[test_index]
            y_train_fold, y_test_fold = y[train_index], y[test_index]

            X_train_tensor = torch.tensor(X_train_fold, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).unsqueeze(1)
            X_test_tensor = torch.tensor(X_test_fold, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test_fold, dtype=torch.float32).unsqueeze(1)

            folds.append((X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor))

        print(f"Train: {folds[0][0].shape[0]}, Test: {folds[0][2].shape[0]}")
        return folds


In [ ]:
irrelevant_columns = ['NumDealsPurchases', 'Response']
marketing_df = marketing_df.drop(columns=irrelevant_columns)

marketing_df['AcceptedAny'] = (
    (marketing_df['AcceptedCmp1'] == 1) |
    (marketing_df['AcceptedCmp2'] == 1) |
    (marketing_df['AcceptedCmp3'] == 1) |
    (marketing_df['AcceptedCmp4'] == 1) |
    (marketing_df['AcceptedCmp5'] == 1)
).astype(int)

marketing_df.drop(columns=['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5'], inplace=True)

print(marketing_df[['AcceptedAny']].value_counts())

In [ ]:
print(marketing_df[['AcceptedAny']].value_counts())
#%%
marketing_df['Education'] = marketing_df[['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD']].idxmax(axis=1)

marketing_df['Marital_Status'] = marketing_df[['Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married',
                                               'Marital_Status_Divorced', 'Marital_Status_Widow', 'Marital_Status_Alone',
                                               'Marital_Status_Other']].idxmax(axis=1)
marketing_df.drop(columns=['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD',
                           'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 'Marital_Status_Divorced',
                           'Marital_Status_Widow', 'Marital_Status_Alone', 'Marital_Status_Other'], inplace=True)

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
marketing_df['Education'] = marketing_df['Education'].str.replace('Education_', '')
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].str.replace('Marital_Status_', '')

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
marketing_df['Education'] = label_enc.fit_transform(marketing_df['Education'])
marketing_df['Marital_Status'] = label_enc.fit_transform(marketing_df['Marital_Status'])

print(marketing_df[['Education', 'Marital_Status']].value_counts())

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')


In [ ]:
def apply_dimensionality_reduction(X_train, X_val, X_test, method, n_components=10):

    if method == 'none':
        # No DR, just return original data
        return X_train, X_val, X_test
    elif method == 'pca':
        dr = PCA(n_components=n_components, random_state=42)
    elif method == 'ica':
        dr = FastICA(n_components=n_components, random_state=42)
    elif method == 'rp':
        dr = GaussianRandomProjection(n_components=n_components, random_state=42)
    else:
        raise ValueError("Unknown DR method")

    dr.fit(X_train)
    X_train_dr = dr.transform(X_train)
    X_val_dr = dr.transform(X_val)
    X_test_dr = dr.transform(X_test)

    return X_train_dr, X_val_dr, X_test_dr

In [ ]:
def run_experiment_with_dr(dr_method, num_epochs=300, lr=0.001):

    X_train_dr, X_val_dr, X_test_dr = apply_dimensionality_reduction(X_train, X_val, X_test, dr_method, n_components=10)

    train_data = torch.utils.data.TensorDataset(
        torch.tensor(X_train_dr, dtype=torch.float32),
        y_train
    )
    val_data = torch.utils.data.TensorDataset(
        torch.tensor(X_val_dr, dtype=torch.float32),
        y_val
    )
    test_data = torch.utils.data.TensorDataset(
        torch.tensor(X_test_dr, dtype=torch.float32),
        y_test
    )

    train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
    val_loader   = torch.utils.data.DataLoader(val_data,   batch_size=64, shuffle=False)
    test_loader  = torch.utils.data.DataLoader(test_data,  batch_size=64, shuffle=False)

    input_dim = X_train_dr.shape[1]
    model = SimpleNN(activation_function=nn.ReLU())
    model.net[0] = nn.Linear(input_dim, 64)

    start_time = time.time()
    train_losses, val_losses, test_losses = train_model(model, train_loader, val_loader, test_loader,
                                                        num_epochs=num_epochs, lr=lr)
    end_time = time.time()
    train_time = end_time - start_time

    final_test_loss = test_losses[-1]
    return final_test_loss, train_time

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, activation_function=nn.ReLU()):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(X_train.shape[1], 64),
            activation_function,
            nn.Linear(64, 32),
            activation_function,
            nn.Linear(32, 16),
            activation_function,
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
num_epochs = 300
batch_size = 64
learning_rate = 0.001

In [ ]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)




In [ ]:

val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train_model(model, train_loader, val_loader, test_loader, num_epochs=200, lr=0.001):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.BCELoss()
    train_losses, val_losses, test_losses = [], [], []
    best_model = None
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                test_loss += loss.item()
        test_loss /= len(test_loader)
        test_losses.append(test_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()

        if best_model:
            torch.save(best_model, "best_model.pth")

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Test Loss: {test_loss:.4f}")

    plt.figure(figsize=(10, 5))
    plt.title('Learning Curve - Neural Network: Loss/Error Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.plot(range(1, num_epochs + 1), train_losses, label='Train Loss')
    plt.plot(range(1, num_epochs + 1), val_losses, label='Val Loss')
    plt.plot(range(1, num_epochs + 1), test_losses, label='Test Loss')
    plt.legend()
    # plt.savefig("loss_all_full_nn_300epoch_400data.png")
    plt.show()



    return train_losses, val_losses, test_losses


In [ ]:
dr_methods = ['none', 'pca', 'ica', 'rp']
final_test_losses = []
training_times = []

for method in dr_methods:
    test_loss, t_time = run_experiment_with_dr(method, num_epochs=300, lr=0.001)
    final_test_losses.append(test_loss)
    training_times.append(t_time)
    print(f"DR={method}: Final Test Loss={test_loss:.4f}, Training Time={t_time:.2f}s")


In [ ]:
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.bar(dr_methods, final_test_losses, color='skyblue', edgecolor='k')
plt.title("Final Test Loss by DR Method")
plt.xlabel("DR Method")
plt.ylabel("Test Loss")

plt.subplot(1,2,2)
plt.bar(dr_methods, training_times, color='coral', edgecolor='k')
plt.title("Training Time by DR Method")
plt.xlabel("DR Method")
plt.ylabel("Seconds")

plt.tight_layout()
plt.show()

In [ ]:
model = SimpleNN(activation_function=nn.ReLU())
start_time = time.time()
train_losses, val_losses, test_losses = train_model(model, train_loader, val_loader, test_loader, num_epochs=num_epochs, lr=0.001)
end_time = time.time()
training_time = end_time - start_time
print(f"Training Time: {training_time:.2f}")

In [ ]:
import numpy as np
import pandas as pd
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# excluding 'AcceptedAny'
orig_features = [col for col in marketing_df.columns if col != 'AcceptedAny']
X_orig = marketing_df[orig_features].values
y_orig = marketing_df['AcceptedAny'].values

scaler = StandardScaler()
X_orig_scaled = scaler.fit_transform(X_orig)

def add_cluster_features(X, method, n_clusters, random_state=42):
    if method == 'kmeans':
        clust = KMeans(n_clusters=n_clusters, random_state=random_state)
        labels = clust.fit_predict(X)
    elif method == 'em':
        gm = GaussianMixture(n_components=n_clusters, random_state=random_state)
        labels = gm.fit_predict(X)
    else:
        raise ValueError("Unknown clustering method")
    encoder = OneHotEncoder(sparse_output=False)
    labels_oh = encoder.fit_transform(labels.reshape(-1, 1))
    X_new = np.hstack([X, labels_oh])
    return X_new

def run_nn_with_clusters(X, y, clustering_method, n_clusters, num_epochs=300, lr=0.001):
    X_aug = add_cluster_features(X, clustering_method, n_clusters, random_state=42)

    new_feature_names = orig_features + [f"{clustering_method.upper()}_{i}" for i in range(n_clusters)]
    df_new = pd.DataFrame(X_aug, columns=new_feature_names)
    df_new['AcceptedAny'] = y

    X_train, y_train, X_val, y_val, X_test, y_test = split_data(df_new, method='train_val_test')

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)

    input_dim = X_train.shape[1]
    model = SimpleNN(activation_function=nn.ReLU())
    model.net[0] = nn.Linear(input_dim, 64)

    start_time = time.time()
    train_losses, val_losses, test_losses = train_model(model, train_loader, val_loader, test_loader,
                                                         num_epochs=num_epochs, lr=lr)
    end_time = time.time()
    training_time = end_time - start_time
    final_test_loss = test_losses[-1]
    return final_test_loss, training_time

baseline_results = {'Final Test Loss': 0.4168, 'Training Time (s)': 36.81}

test_loss_kmeans, time_kmeans = run_nn_with_clusters(X_orig_scaled, y_orig, 'kmeans', n_clusters=5, num_epochs=300, lr=0.001)

test_loss_em, time_em = run_nn_with_clusters(X_orig_scaled, y_orig, 'em', n_clusters=4, num_epochs=300, lr=0.001)

print("Baseline:", baseline_results)
print("KMeans Features:", {"Final Test Loss": test_loss_kmeans, "Training Time (s)": time_kmeans})
print("EM Features:", {"Final Test Loss": test_loss_em, "Training Time (s)": time_em})